# Genie Space Setup

Deploy Velocity Motors Genie Spaces using Infrastructure-as-Code YAML configurations.

## What This Notebook Does
1. Reads YAML config files from `infra/configs/velocity_motors/`
2. Creates Genie Spaces via the Databricks Genie API
3. Outputs deployed space IDs for downstream notebook configuration

## Prerequisites
- Run `00b_setup_data` first (loads data into Unity Catalog)
- Databricks workspace with Unity Catalog enabled
- A SQL Warehouse ID (from the SQL Warehouses page)
- Permission to create Genie Spaces

## Note
The Genie Space Create/Update APIs are in Beta. Join specifications must still be configured manually in the Genie UI after deployment.

## 1. Setup and Environment Check

In [ ]:
# Setup and Environment Check
import os
import sys

IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_DATABRICKS:
    # Path setup for imports
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    workspace_path = "/Workspace" + "/".join(notebook_path.split("/")[:-2])
    if workspace_path not in sys.path:
        sys.path.insert(0, workspace_path)
else:
    project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
    if project_root not in sys.path:
        sys.path.insert(0, project_root)

print("Environment ready!")
print(f"Running in Databricks: {IN_DATABRICKS}")

## 2. Configuration

In [ ]:
# Configuration Widgets
if IN_DATABRICKS:
    try:
        dbutils.widgets.removeAll()
    except Exception:
        pass

    dbutils.widgets.text("1_warehouse_id", "", "1. SQL Warehouse ID")
    dbutils.widgets.text("2_catalog", "workshop", "2. Catalog Name")
    dbutils.widgets.text("3_parent_path", "", "3. Genie Space Parent Path")
    dbutils.widgets.dropdown(
        "4_spaces",
        "domain",
        ["domain", "unified", "all"],
        "4. Spaces to Deploy",
    )

    print("Configure using the widgets above, then run the next cells.")
    print("")
    print("Widget options:")
    print("  1. Warehouse ID: Your SQL Warehouse ID (find in SQL Warehouses page)")
    print("  2. Catalog: The Unity Catalog name used in 00b_setup_data (default: workshop)")
    print("  3. Parent Path: Workspace folder for Genie Spaces")
    print("     Example: /Workspace/Users/your.email@company.com/genie-spaces")
    print("  4. Spaces to Deploy:")
    print("     - domain: 3 domain-specific spaces (Sales, CRM, Operations)")
    print("     - unified: 1 combined space (all 16 tables)")
    print("     - all: All 4 spaces")
else:
    print("Running locally - using default values.")
    print("Modify the execution cell to change configuration.")

## 3. Core Functions

In [ ]:
# Core Functions for Genie Space Deployment
from pathlib import Path

from infra.genie_space_manager import GenieSpaceConfig, GenieSpaceManager

# Space configurations — maps to YAML files in infra/configs/velocity_motors/
SPACE_CONFIGS = {
    "sales": {
        "config_file": "infra/configs/velocity_motors/sales_analytics.yaml",
        "name": "Sales Analytics",
        "description": "Vehicle sales, orders, and salesperson performance",
    },
    "crm": {
        "config_file": "infra/configs/velocity_motors/customer_intelligence.yaml",
        "name": "Customer Intelligence",
        "description": "Customer segments, interactions, and lead pipeline",
    },
    "operations": {
        "config_file": "infra/configs/velocity_motors/operations_inventory.yaml",
        "name": "Operations & Inventory",
        "description": "Parts inventory, suppliers, and service operations",
    },
    "unified": {
        "config_file": "infra/configs/velocity_motors/unified_analytics.yaml",
        "name": "Unified Analytics",
        "description": "Cross-domain analytics combining all 16 tables",
    },
}


def get_base_path():
    """Get the project base path depending on environment."""
    if IN_DATABRICKS:
        return workspace_path
    else:
        return project_root


def preview_deployment(spaces_to_deploy, catalog):
    """Preview what will be deployed (dry-run summary)."""
    print("Deployment Preview")
    print("=" * 60)
    print(f"Catalog: {catalog}")
    print(f"Spaces to deploy: {len(spaces_to_deploy)}")
    print()

    base_path = get_base_path()
    for domain, info in spaces_to_deploy.items():
        config_path = os.path.join(base_path, info["config_file"])
        print(f"  [{domain}] {info['name']}")
        print(f"    Config: {info['config_file']}")
        print(f"    Description: {info['description']}")

        # Load config to show table count
        try:
            config = GenieSpaceConfig.from_yaml(config_path)
            print(f"    Tables: {len(config.tables)}")
        except Exception as e:
            print(f"    (Could not load config: {e})")
        print()


def deploy_spaces(spaces_to_deploy, dry_run=False):
    """Deploy Genie Spaces from YAML configs.

    Args:
        spaces_to_deploy: Dict of domain -> space config info
        dry_run: If True, preview without creating spaces

    Returns:
        Dict of domain -> {"space_id": str, "name": str, "success": bool}
    """
    manager = GenieSpaceManager()
    results = {}
    base_path = get_base_path()

    for domain, info in spaces_to_deploy.items():
        config_path = os.path.join(base_path, info["config_file"])
        print(f"\n{'=' * 60}")
        print(f"Deploying: {info['name']}")
        print(f"Config: {info['config_file']}")

        if not os.path.exists(config_path):
            print(f"  ERROR: Config file not found: {config_path}")
            results[domain] = {"space_id": None, "name": info["name"], "success": False}
            continue

        try:
            space_id = manager.deploy_from_config(config_path, dry_run=dry_run)
            results[domain] = {"space_id": space_id, "name": info["name"], "success": True}
            if dry_run:
                print(f"  [DRY RUN] Would create space: {info['name']}")
            else:
                print(f"  Space created! ID: {space_id}")
        except Exception as e:
            print(f"  ERROR: {e}")
            results[domain] = {"space_id": None, "name": info["name"], "success": False}

    return results


print("Core functions loaded!")

## 4. Deploy Genie Spaces

This cell will:
1. Validate configuration (warehouse ID and parent path required)
2. Set environment variables for YAML config substitution
3. Preview deployment plan
4. Create Genie Spaces via the API

In [ ]:
# Deploy Genie Spaces

# Get configuration
if IN_DATABRICKS:
    warehouse_id = dbutils.widgets.get("1_warehouse_id")
    catalog = dbutils.widgets.get("2_catalog")
    parent_path = dbutils.widgets.get("3_parent_path")
    spaces_option = dbutils.widgets.get("4_spaces")
else:
    warehouse_id = os.environ.get("WAREHOUSE_ID", "")
    catalog = os.environ.get("CATALOG", "workshop")
    parent_path = os.environ.get("PARENT_PATH", "")
    spaces_option = "domain"

print("Configuration:")
print(f"  Warehouse ID: {warehouse_id or '(not set)'}")
print(f"  Catalog: {catalog}")
print(f"  Parent Path: {parent_path or '(not set)'}")
print(f"  Spaces: {spaces_option}")
print()

if not IN_DATABRICKS:
    print("NOTICE: Not running in Databricks. Skipping deployment.")
    print("This notebook is designed to run in Databricks.")
else:
    # Validate required fields
    if not warehouse_id:
        print("ERROR: Warehouse ID is required.")
        print("  Set it in widget '1. SQL Warehouse ID' above.")
        print("  Find your Warehouse ID: SQL Warehouses page > click warehouse > copy ID from URL")
        raise ValueError("Warehouse ID is required")

    if not parent_path:
        print("ERROR: Parent Path is required.")
        print("  Set it in widget '3. Genie Space Parent Path' above.")
        print("  Example: /Workspace/Users/your.email@company.com/genie-spaces")
        raise ValueError("Parent Path is required")

    # Set environment variables for YAML ${VAR} substitution
    os.environ["WAREHOUSE_ID"] = warehouse_id
    os.environ["PARENT_PATH"] = parent_path

    # Determine which spaces to deploy
    if spaces_option == "domain":
        spaces_to_deploy = {k: v for k, v in SPACE_CONFIGS.items() if k != "unified"}
    elif spaces_option == "unified":
        spaces_to_deploy = {"unified": SPACE_CONFIGS["unified"]}
    else:  # all
        spaces_to_deploy = SPACE_CONFIGS.copy()

    # Preview
    preview_deployment(spaces_to_deploy, catalog)

    # Deploy
    print("\nStarting deployment...")
    print("-" * 60)
    results = deploy_spaces(spaces_to_deploy)

    # Summary
    print("\n" + "=" * 60)
    print("DEPLOYMENT SUMMARY")
    print("=" * 60)

    success_count = sum(1 for r in results.values() if r["success"])
    failed_count = sum(1 for r in results.values() if not r["success"])

    for domain, result in results.items():
        status = "OK" if result["success"] else "FAILED"
        space_id = result["space_id"] or "N/A"
        print(f"  [{status}] {result['name']}: {space_id}")

    print(f"\nTotal: {success_count} deployed, {failed_count} failed")

## 5. Verify Deployment

In [ ]:
# Verify deployed spaces

if IN_DATABRICKS and 'results' in dir():
    print("Deployment Verification")
    print("=" * 60)

    manager = GenieSpaceManager()
    for domain, result in results.items():
        if result["success"] and result["space_id"]:
            try:
                space = manager.get_space(result["space_id"])
                print(f"\n  [{domain}] {result['name']}")
                print(f"    Space ID: {result['space_id']}")
                print(f"    Title: {space.get('title', 'N/A')}")
                print(f"    Status: Active")
            except Exception as e:
                print(f"\n  [{domain}] {result['name']}")
                print(f"    WARNING: Could not verify - {e}")
        elif not result["success"]:
            print(f"\n  [{domain}] {result['name']}: Skipped (deployment failed)")

    print("\nVerification complete!")
else:
    print("Verification skipped - deploy spaces first (run cells above).")

## 6. Next Steps

Configure your notebooks with the deployed space IDs:

1. **Notebook 01** (`01_agent_basics.ipynb`): Set the `GENIE_SPACE_ID` widget to any single space ID above
2. **Notebook 02** (`02_multi_genie_orchestration.ipynb`): Uses the 3 domain-specific spaces for multi-Genie demo

### Manual UI Steps (Beta API Limitation)
After deployment, configure join specifications in each Genie Space UI:
1. Open each space in the Genie UI
2. Go to Settings > Data
3. Add join keys between related tables (see `infra/configs/velocity_motors/*.yaml` for join specs)

In [ ]:
# Space IDs for Configuration

if IN_DATABRICKS and 'results' in dir():
    print("Copy these space IDs into your notebook widgets:")
    print()

    for domain, result in results.items():
        if result["success"] and result["space_id"]:
            print(f"  {result['name']}: {result['space_id']}")

    print()
    print("For notebook 02 (Multi-Genie Orchestration):")
    domain_spaces = {d: r for d, r in results.items() if r["success"] and d != "unified"}
    if domain_spaces:
        ids = [r["space_id"] for r in domain_spaces.values() if r["space_id"]]
        print(f"  Space IDs (comma-separated): {','.join(ids)}")
else:
    print("Deploy spaces first (run cells above).")

## 7. Cleanup (Optional)

Uncomment and run the cell below to delete deployed Genie Spaces.
**WARNING**: This cannot be undone!

In [ ]:
# CLEANUP - Uncomment to delete deployed Genie Spaces
# WARNING: This will permanently delete the spaces!

# if IN_DATABRICKS and 'results' in dir():
#     manager = GenieSpaceManager()
#     for domain, result in results.items():
#         if result["success"] and result["space_id"]:
#             print(f"Deleting {result['name']} ({result['space_id']})...")
#             try:
#                 manager.delete_space(result["space_id"])
#                 print(f"  Deleted!")
#             except Exception as e:
#                 print(f"  ERROR: {e}")
# else:
#     print("No spaces to clean up.")